In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.pipeline import Pipeline
from pathlib import Path
import sys
base_dir = Path().resolve().parent
sys.path.append(str(base_dir / "src"))
from helpers.tree_data import tree_data
import warnings
warnings.filterwarnings("ignore")
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [2]:
dataset = base_dir / "data" / "telco_customer_churn_clean.csv"
X_train, X_test, y_train, y_test = tree_data(dataset)

Dataset 'C:\Users\HP\Desktop\Telco_Customer_Churn\data\telco_customer_churn_clean.csv' loaded successfully.

TotalCharges was typecasted to numerical.

Null values were removed.


Dataset size : 
7032 rows
21 columns


Successfully dropped the column: 'customerID'
Dataset split complete.

Successfully encoded all columns


Training and testing datasets are ready to be used.



In [3]:
max_depths = [3, 5, 7, 10, None]
min_samples_leafs = [1, 2, 5, 10, 20]
metrics = ["recall", "precision", "f1", "accuracy"]

new_arr = np.zeros((25,4))
new_df = pd.DataFrame(
    new_arr,
    index=pd.MultiIndex.from_product(
        [max_depths, min_samples_leafs],
        names=["max_depth", "min_samples_leafs"]
    ),
    columns=metrics
)

scores = {}

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        model = DecisionTreeClassifier(
            max_depth = max_depth,
            min_samples_leaf = min_samples_leaf,
            criterion = "gini",
            class_weight = None,
            random_state = 42
        )

        skf = StratifiedKFold(
            n_splits = 5,
            shuffle = True,
            random_state = 42
        )

        model_scores = cross_validate(
            estimator = model,
            scoring = metrics,
            X = X_train,
            y = y_train,
            cv = skf
        )

        scores[(max_depth,min_samples_leaf)] = model_scores

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        print(f"Max Depth : {max_depth} | min_samples_leaf : {min_samples_leaf}")
        curr_dict = scores[(max_depth,min_samples_leaf)]
        for metric in metrics:
            mean_score = curr_dict[f"test_{metric}"].mean()
            print(f"Metric : {metric} | Mean Score : {mean_score}")
            new_df.loc[(max_depth,min_samples_leaf),metric] = mean_score
        print()

new_df.to_csv(base_dir / "data" / "DT_Baseline_DiffParams.csv")

Max Depth : 3 | min_samples_leaf : 1
Metric : recall | Mean Score : 0.3612040133779264
Metric : precision | Mean Score : 0.7027878354314654
Metric : f1 | Mean Score : 0.4763226367049387
Metric : accuracy | Mean Score : 0.7893333333333332

Max Depth : 3 | min_samples_leaf : 2
Metric : recall | Mean Score : 0.3612040133779264
Metric : precision | Mean Score : 0.7027878354314654
Metric : f1 | Mean Score : 0.4763226367049387
Metric : accuracy | Mean Score : 0.7893333333333332

Max Depth : 3 | min_samples_leaf : 5
Metric : recall | Mean Score : 0.3612040133779264
Metric : precision | Mean Score : 0.7027878354314654
Metric : f1 | Mean Score : 0.4763226367049387
Metric : accuracy | Mean Score : 0.7893333333333332

Max Depth : 3 | min_samples_leaf : 10
Metric : recall | Mean Score : 0.3612040133779264
Metric : precision | Mean Score : 0.7027878354314654
Metric : f1 | Mean Score : 0.4763226367049387
Metric : accuracy | Mean Score : 0.7893333333333332

Max Depth : 3 | min_samples_leaf : 20
Metri

In [4]:
max_depths = [3, 5, 7, 10, None]
min_samples_leafs = [1, 2, 5, 10, 20]
metrics = ["recall", "precision", "f1", "accuracy"]

new_arr = np.zeros((25,4))
new_df = pd.DataFrame(
    new_arr,
    index=pd.MultiIndex.from_product(
        [max_depths, min_samples_leafs],
        names=["max_depth", "min_samples_leafs"]
    ),
    columns=metrics
)

scores = {}

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        model = DecisionTreeClassifier(
            max_depth = max_depth,
            min_samples_leaf = min_samples_leaf,
            criterion = "gini",
            class_weight = "balanced",
            random_state = 42
        )

        skf = StratifiedKFold(
            n_splits = 5,
            shuffle = True,
            random_state = 42
        )

        model_scores = cross_validate(
            estimator = model,
            scoring = metrics,
            X = X_train,
            y = y_train,
            cv = skf
        )

        scores[(max_depth,min_samples_leaf)] = model_scores

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        print(f"Max Depth : {max_depth} | min_samples_leaf : {min_samples_leaf}")
        curr_dict = scores[(max_depth,min_samples_leaf)]
        for metric in metrics:
            mean_score = curr_dict[f"test_{metric}"].mean()
            print(f"Metric : {metric} | Mean Score : {mean_score}")
            new_df.loc[(max_depth,min_samples_leaf),metric] = mean_score
        print()

new_df.to_csv(base_dir / "data" / "DT_Baseline_DiffParams_balanced.csv")

Max Depth : 3 | min_samples_leaf : 1
Metric : recall | Mean Score : 0.7819397993311037
Metric : precision | Mean Score : 0.5170772888905363
Metric : f1 | Mean Score : 0.6223381492377056
Metric : accuracy | Mean Score : 0.7479111111111111

Max Depth : 3 | min_samples_leaf : 2
Metric : recall | Mean Score : 0.7819397993311037
Metric : precision | Mean Score : 0.5170772888905363
Metric : f1 | Mean Score : 0.6223381492377056
Metric : accuracy | Mean Score : 0.7479111111111111

Max Depth : 3 | min_samples_leaf : 5
Metric : recall | Mean Score : 0.7819397993311037
Metric : precision | Mean Score : 0.5170772888905363
Metric : f1 | Mean Score : 0.6223381492377056
Metric : accuracy | Mean Score : 0.7479111111111111

Max Depth : 3 | min_samples_leaf : 10
Metric : recall | Mean Score : 0.7819397993311037
Metric : precision | Mean Score : 0.5170772888905363
Metric : f1 | Mean Score : 0.6223381492377056
Metric : accuracy | Mean Score : 0.7479111111111111

Max Depth : 3 | min_samples_leaf : 20
Metri

In [5]:
max_depths = [3, 5, 7, 10, None]
min_samples_leafs = [1, 2, 5, 10, 20]
metrics = ["recall", "precision", "f1", "accuracy"]

new_arr = np.zeros((25,4))
new_df = pd.DataFrame(
    new_arr,
    index=pd.MultiIndex.from_product(
        [max_depths, min_samples_leafs],
        names=["max_depth", "min_samples_leafs"]
    ),
    columns=metrics
)

scores = {}

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        model = DecisionTreeClassifier(
            max_depth = max_depth,
            min_samples_leaf = min_samples_leaf,
            criterion = "gini",
            class_weight = None,
            random_state = 42
        )

        skf = StratifiedKFold(
            n_splits = 5,
            shuffle = True,
            random_state = 42
        )

        smotetomek = SMOTETomek(
            random_state=42
        )

        pipeline = Pipeline([
            ("smote", smotetomek),
            ("model", model)
        ])

        model_scores = cross_validate(
            estimator = pipeline,
            scoring = metrics,
            X = X_train,
            y = y_train,
            cv = skf
        )

        scores[(max_depth,min_samples_leaf)] = model_scores

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        print(f"Max Depth : {max_depth} | min_samples_leaf : {min_samples_leaf}")
        curr_dict = scores[(max_depth,min_samples_leaf)]
        for metric in metrics:
            mean_score = curr_dict[f"test_{metric}"].mean()
            print(f"Metric : {metric} | Mean Score : {mean_score}")
            new_df.loc[(max_depth,min_samples_leaf),metric] = mean_score
        print()

new_df.to_csv(base_dir / "data" / "DT_SMOTETomek_DiffParams.csv")

Max Depth : 3 | min_samples_leaf : 1
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 2
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 5
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 10
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 20
Metric : 

In [6]:
max_depths = [3, 5, 7, 10, None]
min_samples_leafs = [1, 2, 5, 10, 20]
metrics = ["recall", "precision", "f1", "accuracy"]

new_arr = np.zeros((25,4))
new_df = pd.DataFrame(
    new_arr,
    index=pd.MultiIndex.from_product(
        [max_depths, min_samples_leafs],
        names=["max_depth", "min_samples_leafs"]
    ),
    columns=metrics
)

scores = {}

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        model = DecisionTreeClassifier(
            max_depth = max_depth,
            min_samples_leaf = min_samples_leaf,
            criterion = "gini",
            class_weight = "balanced",
            random_state = 42
        )

        skf = StratifiedKFold(
            n_splits = 5,
            shuffle = True,
            random_state = 42
        )

        smotetomek = SMOTETomek(
            random_state=42
        )

        pipeline = Pipeline([
            ("smote", smotetomek),
            ("model", model)
        ])

        model_scores = cross_validate(
            estimator = pipeline,
            scoring = metrics,
            X = X_train,
            y = y_train,
            cv = skf
        )

        scores[(max_depth,min_samples_leaf)] = model_scores

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        print(f"Max Depth : {max_depth} | min_samples_leaf : {min_samples_leaf}")
        curr_dict = scores[(max_depth,min_samples_leaf)]
        for metric in metrics:
            mean_score = curr_dict[f"test_{metric}"].mean()
            print(f"Metric : {metric} | Mean Score : {mean_score}")
            new_df.loc[(max_depth,min_samples_leaf),metrics] = mean_score
        print()

new_df.to_csv(base_dir / "data" / "DT_SMOTETomek_DiffParams_balanced.csv")

Max Depth : 3 | min_samples_leaf : 1
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 2
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 5
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 10
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 20
Metric : 

In [7]:
max_depths = [3, 5, 7, 10, None]
min_samples_leafs = [1, 2, 5, 10, 20]
metrics = ["recall", "precision", "f1", "accuracy"]

new_arr = np.zeros((25,4))
new_df = pd.DataFrame(
    new_arr,
    index=pd.MultiIndex.from_product(
        [max_depths, min_samples_leafs],
        names=["max_depth", "min_samples_leafs"]
    ),
    columns=metrics
)

scores = {}

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        model = DecisionTreeClassifier(
            max_depth = max_depth,
            min_samples_leaf = min_samples_leaf,
            criterion = "gini",
            class_weight = "balanced",
            random_state = 42
        )

        skf = StratifiedKFold(
            n_splits = 5,
            shuffle = True,
            random_state = 42
        )

        smotenc = SMOTENC(
            random_state=42,
            categorical_features=[
                col for col in X_train.columns.tolist() if col not in (
                    "tenure",
                    "MonthlyCharges",
                    "TotalCharges"
                )
            ]
        )

        pipeline = Pipeline([
            ("smote", smotetomek),
            ("model", model)
        ])

        model_scores = cross_validate(
            estimator = pipeline,
            scoring = metrics,
            X = X_train,
            y = y_train,
            cv = skf
        )

        scores[(max_depth,min_samples_leaf)] = model_scores

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leafs:
        print(f"Max Depth : {max_depth} | min_samples_leaf : {min_samples_leaf}")
        curr_dict = scores[(max_depth,min_samples_leaf)]
        for metric in metrics:
            mean_score = curr_dict[f"test_{metric}"].mean()
            print(f"Metric : {metric} | Mean Score : {mean_score}")
            new_df.loc[(max_depth,min_samples_leaf),metrics] = mean_score
        print()

new_df.to_csv(base_dir / "data" / "DT_SMOTENC_DiffParams_balanced.csv")

Max Depth : 3 | min_samples_leaf : 1
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 2
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 5
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 10
Metric : recall | Mean Score : 0.6675585284280936
Metric : precision | Mean Score : 0.5183891064805961
Metric : f1 | Mean Score : 0.5831861799213611
Metric : accuracy | Mean Score : 0.746488888888889

Max Depth : 3 | min_samples_leaf : 20
Metric : 

In [8]:
n_estimators = [50, 100, 200, 300]
max_depths = [3, 5, 7, 10]
min_samples_leafs = [2, 5, 10, 20]
metrics = ["recall", "precision", "f1", "accuracy"]

new_arr = np.zeros((64,4))
new_df = pd.DataFrame(
    new_arr,
    columns=metrics,
    index=pd.MultiIndex.from_product(
        [n_estimators,max_depths,min_samples_leafs],
        names=["n_estimatos","max_depths","min_samples_leafs"]
    )
)

scores = {}

for n_estimator in n_estimators:
    for max_depth in max_depths:
        for min_samples_leaf in min_samples_leafs:

            model = RandomForestClassifier(
                n_estimators=n_estimator,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                criterion="gini",
                class_weight="balanced",
                max_features="sqrt"
            )

            skf = StratifiedKFold(
                n_splits=5,
                random_state=42,
                shuffle=True
            )

            model_scores = cross_validate(
                estimator=model,
                cv=skf,
                X=X_train,
                y=y_train,
                scoring=metrics
            )

            scores[(n_estimator,max_depth,min_samples_leaf)] = model_scores

for n_estimator in n_estimators:
    for max_depth in max_depths:
        for min_samples_leaf in min_samples_leafs:
            print(f"estimators = {n_estimator} | depth = {max_depth} | leaf samples = {min_samples_leaf}")
            curr_dict = scores[(n_estimator,max_depth,min_samples_leaf)]
            for metric in metrics:
                mean_score = curr_dict[f"test_{metric}"].mean()
                print(f"Metric : {metric} | Mean Score : {mean_score}")
                new_df.loc[(n_estimator,max_depth,min_samples_leaf),metric]=mean_score

new_df.to_csv(base_dir / "data" / "RFC_DiffParams.csv")

estimators = 50 | depth = 3 | leaf samples = 2
Metric : recall | Mean Score : 0.8321070234113712
Metric : precision | Mean Score : 0.4919301816860958
Metric : f1 | Mean Score : 0.6181546567355736
Metric : accuracy | Mean Score : 0.7267555555555556
estimators = 50 | depth = 3 | leaf samples = 5
Metric : recall | Mean Score : 0.8254180602006688
Metric : precision | Mean Score : 0.5010340807358292
Metric : f1 | Mean Score : 0.6234283922454533
Metric : accuracy | Mean Score : 0.7349333333333332
estimators = 50 | depth = 3 | leaf samples = 10
Metric : recall | Mean Score : 0.8334448160535117
Metric : precision | Mean Score : 0.48662599952097574
Metric : f1 | Mean Score : 0.6141640652657119
Metric : accuracy | Mean Score : 0.7216000000000001
estimators = 50 | depth = 3 | leaf samples = 20
Metric : recall | Mean Score : 0.825418060200669
Metric : precision | Mean Score : 0.4984643084788029
Metric : f1 | Mean Score : 0.621356595453152
Metric : accuracy | Mean Score : 0.7324444444444445
estimat

In [10]:
n_estimators = [100, 200]
learning_rates = [0.003, 0.009, 0.01, 0.03]
max_depths = [5,7]
min_child_weights = [5, 10, 20]

metrics = ["recall", "precision", "f1", "accuracy"]

n_rows = len(n_estimators) * len(learning_rates) * len(max_depths) * len(min_child_weights)
n_cols = len(metrics)

new_arr = np.zeros((n_rows,n_cols))
new_df = pd.DataFrame(
    new_arr,
    index = pd.MultiIndex.from_product(
        [n_estimators, learning_rates, max_depths, min_child_weights],
        names = ["n_estimators", "learning_rate", "max_depth", "min_child_weights"]
    ),
    columns = metrics
)

models = {}

for n_estimator in n_estimators:
    for learning_rate in learning_rates:
        for max_depth in max_depths:
            for min_child_weight in min_child_weights:

                model = XGBClassifier(
                    n_estimators = n_estimator,
                    learning_rate = learning_rate,
                    max_depth = max_depth,
                    min_child_weight = min_child_weight,
                    subsample = 0.8,
                    colsample_bytree = 0.7,
                    scale_pos_weight = 3,
                    random_state = 42,
                    verbosity = 1
                )

                skf = StratifiedKFold(
                    n_splits = 5,
                    shuffle = True,
                    random_state = 42
                )

                scores = cross_validate(
                    estimator = model,
                    X = X_train,
                    y = y_train,
                    scoring = metrics,
                    cv = skf
                )

                models[(n_estimator,learning_rate,max_depth,min_child_weight)] = scores

for n_estimator in n_estimators:
    for learning_rate in learning_rates:
        for max_depth in max_depths:
            for min_child_weight in min_child_weights:
                print(f"n_estimators : {n_estimator}, | learning_rate : {learning_rate} | max_depth : {max_depth} | min_child_weight : {min_child_weight}")
                curr_dict = models[(n_estimator,learning_rate,max_depth,min_child_weight)]
                for metric in metrics:
                    mean_score = curr_dict[f"test_{metric}"].mean()
                    print(f"Metric : {metric} | Mean Score : {mean_score}")
                    new_df.loc[(n_estimator,learning_rate,max_depth,min_child_weight),metric] = mean_score
                print()

new_df.to_csv(base_dir / "data" / "XGBC_DiffParams.csv")

n_estimators : 100, | learning_rate : 0.003 | max_depth : 5 | min_child_weight : 5
Metric : recall | Mean Score : 0.85752508361204
Metric : precision | Mean Score : 0.4886954507469225
Metric : f1 | Mean Score : 0.6225370066341654
Metric : accuracy | Mean Score : 0.7235555555555555

n_estimators : 100, | learning_rate : 0.003 | max_depth : 5 | min_child_weight : 10
Metric : recall | Mean Score : 0.85752508361204
Metric : precision | Mean Score : 0.48852901176647145
Metric : f1 | Mean Score : 0.6223888674752167
Metric : accuracy | Mean Score : 0.7233777777777777

n_estimators : 100, | learning_rate : 0.003 | max_depth : 5 | min_child_weight : 20
Metric : recall | Mean Score : 0.8561872909698997
Metric : precision | Mean Score : 0.48647940816862595
Metric : f1 | Mean Score : 0.6203684735743663
Metric : accuracy | Mean Score : 0.7214222222222222

n_estimators : 100, | learning_rate : 0.003 | max_depth : 7 | min_child_weight : 5
Metric : recall | Mean Score : 0.8488294314381271
Metric : pre

In [15]:
n_estimators = [150, 200]
learning_rates = [0.01, 0.03, 0.09]
max_depths = [5,7]
min_child_samples = [5, 10, 20]
num_leaves = [31, 62, 124]
metrics = ["recall", "precision", "f1", "accuracy"]

n_rows = len(n_estimators) * len(learning_rates) * len(max_depths) * len(min_child_samples) * len(num_leaves)
n_cols = len(metrics)

new_arr = np.zeros((n_rows,n_cols))
new_df = pd.DataFrame(
    new_arr,
    index = pd.MultiIndex.from_product(
        [n_estimators, learning_rates, max_depths, min_child_samples, num_leaves],
        names = ["n_estimators", "learning_rate", "max_depth", "min_child_samples", "num_leaves"]
    ),
    columns = metrics
)

models = {}

mix = 1

for n_estimator in n_estimators:
    for learning_rate in learning_rates:
        for max_depth in max_depths:
            for min_child_sample in min_child_samples:
                for num_leaf in num_leaves:

                    print(f"\nModel Set {mix}\n")
                    print(f"estimators : {n_estimator} | learning_rate : {learning_rate} | max_depth : {max_depth} | min_child_samples : {min_child_sample} | num_leaves : {num_leaf}")

                    model = LGBMClassifier(
                        n_estimators = n_estimator,
                        learning_rate = learning_rate,
                        max_depth = max_depth,
                        num_leaves = num_leaf,
                        subsample = 0.8,
                        colsample_bytree = 0.7,
                        scale_pos_weight = 3,
                        random_state = 42,
                        objective = "binary",
                        verbosity = -1
                        
                    )

                    skf = StratifiedKFold(
                        n_splits = 5,
                        shuffle = True,
                        random_state = 42
                    )

                    scores = cross_validate(
                        estimator = model,
                        cv = skf,
                        X = X_train,
                        y = y_train,
                        scoring = metrics
                    )

                    models[(n_estimator, learning_rate, max_depth, min_child_sample, num_leaf)] = scores

                    mix += 1

for n_estimator in n_estimators:
    for learning_rate in learning_rates:
        for max_depth in max_depths:
            for min_child_sample in min_child_samples:
                for num_leaf in num_leaves:
                    print(f"estimators : {n_estimator} | learning_rate : {learning_rate} | max_depth : {max_depth} | min_child_samples : {min_child_sample} | num_leaves : {num_leaf}")
                    curr_dict = models[(n_estimator, learning_rate, max_depth, min_child_sample, num_leaf)]

                    for metric in metrics:

                        mean_score = curr_dict[f"test_{metric}"].mean()
                        new_df.loc[(n_estimator, learning_rate, max_depth, min_child_sample, num_leaf),metric] = mean_score
                        print(f"Metric : {metric} | Mean Score : {mean_score}")
                    print()

new_df.to_csv(base_dir / "data" / "LGBM_DiffParams.csv")


Model Set 1

estimators : 150 | learning_rate : 0.01 | max_depth : 5 | min_child_samples : 5 | num_leaves : 31

Model Set 2

estimators : 150 | learning_rate : 0.01 | max_depth : 5 | min_child_samples : 5 | num_leaves : 62

Model Set 3

estimators : 150 | learning_rate : 0.01 | max_depth : 5 | min_child_samples : 5 | num_leaves : 124

Model Set 4

estimators : 150 | learning_rate : 0.01 | max_depth : 5 | min_child_samples : 10 | num_leaves : 31

Model Set 5

estimators : 150 | learning_rate : 0.01 | max_depth : 5 | min_child_samples : 10 | num_leaves : 62

Model Set 6

estimators : 150 | learning_rate : 0.01 | max_depth : 5 | min_child_samples : 10 | num_leaves : 124

Model Set 7

estimators : 150 | learning_rate : 0.01 | max_depth : 5 | min_child_samples : 20 | num_leaves : 31

Model Set 8

estimators : 150 | learning_rate : 0.01 | max_depth : 5 | min_child_samples : 20 | num_leaves : 62

Model Set 9

estimators : 150 | learning_rate : 0.01 | max_depth : 5 | min_child_samples : 20 | 

In [29]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, cv, Pool
from pathlib import Path
import sys
base_dir = Path().resolve().parent
sys.path.append(str(base_dir / "src"))
from helpers.cat_boost_data import cat_boost_data

dataset = base_dir / "data" / "telco_customer_churn_clean.csv"
X_train, X_test, y_train, y_test, cat_cols = cat_boost_data(dataset)

iterations = [100, 200, 300]
learning_rates = [0.003, 0.009, 0.03]
depths = [5, 7]
l2_leaf_regs = [2.0, 3.0, 4.0]
metrics = ["test-Recall-mean", "train-Recall-mean"]

n_rows = len(iterations) * len(learning_rates) * len(depths) * len(l2_leaf_regs)
n_cols = len(metrics)

new_arr = np.zeros((n_rows,n_cols))
new_df = pd.DataFrame(
    new_arr,
    index=pd.MultiIndex.from_product(
        [iterations,learning_rates,depths,l2_leaf_regs],
        names=["iterations", "learning_rates", "depths", "l2_leaf_regs"]
    ),
    columns=metrics
)

models = {}

no = 1

for iteration in iterations:
    for learning_rate in learning_rates:
        for depth in depths:
            for l2_leaf_reg in l2_leaf_regs:

                train_pool = Pool(
                    data = X_train,
                    label = y_train,
                    cat_features = cat_cols
                )

                params = {
                    "iterations":iteration,
                    "learning_rate":learning_rate,
                    "depth":depth,
                    "l2_leaf_reg":l2_leaf_reg,
                    "loss_function":"Logloss",
                    "eval_metric":"Recall",
                    "custom_metric":[
                        "Precision",
                        "F1",
                        "Accuracy"
                    ],
                    "scale_pos_weight":3,
                    "random_seed":42,
                    "verbose": 50,
                    "task_type":"GPU"
                }

                cv_results = cv(
                    pool = train_pool,
                    params = params,
                    fold_count = 5,
                    stratified = True,
                    shuffle = True,
                    partition_random_seed = 42,
                    early_stopping_rounds = 30
                )

                models[(iteration,learning_rate,depth,l2_leaf_reg)] = cv_results

                print(f"Model {no} Trained\n")

                no += 1



Dataset 'C:\Users\HP\Desktop\Telco_Customer_Churn\data\telco_customer_churn_clean.csv' loaded successfully.

TotalCharges Column Dropped.

All null values were dropped.
customerID column was dropped.

Successfully encoded the SeniorCitizen column.

Successfully identified categorical columns.
Dataset Split complete.

Returned X_train, X_test, y_train, y_test, cat_cols
Training on fold [0/5]
0:	learn: 0.8235786	test: 0.8327759	best: 0.8327759 (0)	total: 54.9ms	remaining: 5.43s
50:	learn: 0.8076923	test: 0.8428094	best: 0.8494983 (42)	total: 3.86s	remaining: 3.71s
bestTest = 0.8494983278
bestIteration = 42
Training on fold [1/5]
0:	learn: 0.8277592	test: 0.8127090	best: 0.8127090 (0)	total: 68.7ms	remaining: 6.8s
bestTest = 0.8193979933
bestIteration = 1
Training on fold [2/5]
0:	learn: 0.8436455	test: 0.8461538	best: 0.8461538 (0)	total: 65.6ms	remaining: 6.49s
bestTest = 0.8461538462
bestIteration = 0
Training on fold [3/5]
0:	learn: 0.8603679	test: 0.8193980	best: 0.8193980 (0)	total:

NameError: name 'curr_dict' is not defined

In [32]:
for iteration in iterations:
    for learning_rate in learning_rates:
        for depth in depths:
            for l2_leaf_reg in l2_leaf_regs:
                print(f"iteration : {iteration} | learning_rate {learning_rate} | depth : {depth} | l2_leaf_reg : {l2_leaf_reg}")
                curr_df = models[(iteration,learning_rate,depth,l2_leaf_reg)]
                for col in curr_df.columns.to_list():
                    if col in metrics:
                        mean_score = curr_df[col].mean()
                        new_df.loc[(iteration,learning_rate,depth,l2_leaf_reg), col] = mean_score
                        print(f"Metric : {col} | Score : {mean_score}")
                print()

new_df.to_csv(base_dir / "data" / "CB_DiffParams.csv")

iteration : 100 | learning_rate 0.003 | depth : 5 | l2_leaf_reg : 2.0
Metric : test-Recall-mean | Score : 0.8133779264214047
Metric : train-Recall-mean | Score : 0.8180327117789893

iteration : 100 | learning_rate 0.003 | depth : 5 | l2_leaf_reg : 3.0
Metric : test-Recall-mean | Score : 0.8157649681946357
Metric : train-Recall-mean | Score : 0.8191258443176602

iteration : 100 | learning_rate 0.003 | depth : 5 | l2_leaf_reg : 4.0
Metric : test-Recall-mean | Score : 0.8155183946488295
Metric : train-Recall-mean | Score : 0.8199916387959865

iteration : 100 | learning_rate 0.003 | depth : 7 | l2_leaf_reg : 2.0
Metric : test-Recall-mean | Score : 0.8139764126034147
Metric : train-Recall-mean | Score : 0.8211406442527722

iteration : 100 | learning_rate 0.003 | depth : 7 | l2_leaf_reg : 3.0
Metric : test-Recall-mean | Score : 0.8134325302027168
Metric : train-Recall-mean | Score : 0.8203637976929902

iteration : 100 | learning_rate 0.003 | depth : 7 | l2_leaf_reg : 4.0
Metric : test-Recall